<a href="https://colab.research.google.com/github/sameerkumyadav-byte/ai-risk-manager-fraud-detector/blob/main/fraud_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# This loads a copy of the Kaggle Credit Card Fraud dataset
url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
print(df['Class'].value_counts())

Class
0    284315
1       492
Name: count, dtype: int64


In [ ]:
df.shape


(284807, 31)

In [ ]:
df['Class'].value_counts()

,count
Class,
0,284315
1,492


In [ ]:
# Separate fraud and normal transactions
fraud = df[df['Class'] == 1]
normal = df[df['Class'] == 0]

print("Fraud transactions - Amount stats:")
print(fraud['Amount'].describe())

print("\nNormal transactions - Amount stats:")
print(normal['Amount'].describe())

Fraud transactions - Amount stats:
count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64

Normal transactions - Amount stats:
count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

# X = features (everything except the label)
# y = label (Class column - what we want to predict)
X = df.drop('Class', axis=1)
y = df['Class']

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Fraud in training:", y_train.sum())
print("Fraud in testing:", y_test.sum())

Training rows: 227845
Testing rows: 56962
Fraud in training: 394
Fraud in testing: 98


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the model
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Train it on the training data
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Ask the model to predict on data it has NEVER seen (X_test)
y_pred = model.predict(X_test)

# Compare predictions to the real answers
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[56861     3]
 [   25    73]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.74      0.84        98

    accuracy                           1.00     56962
   macro avg       0.98      0.87      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [ ]:
import numpy as np

# Instead of a hard 0/1 prediction, get the probability of fraud
y_probs = model.predict_proba(X_test)[:, 1]

# Try a few different thresholds
for threshold in [0.2, 0.3, 0.5, 0.7]:
    y_pred_t = (y_probs >= threshold).astype(int)
    print(f"\n--- Threshold: {threshold} ---")
    print(confusion_matrix(y_test, y_pred_t))
    print(classification_report(y_test, y_pred_t, digits=3))



--- Threshold: 0.2 ---
[[56850    14]
 [   14    84]]
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56864
           1      0.857     0.857     0.857        98

    accuracy                          1.000     56962
   macro avg      0.928     0.928     0.928     56962
weighted avg      1.000     1.000     1.000     56962


--- Threshold: 0.3 ---
[[56857     7]
 [   15    83]]
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56864
           1      0.922     0.847     0.883        98

    accuracy                          1.000     56962
   macro avg      0.961     0.923     0.941     56962
weighted avg      1.000     1.000     1.000     56962


--- Threshold: 0.5 ---
[[56861     3]
 [   25    73]]
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56864
           1      0.961     0.745     0.839        98

    accuracy     

In [ ]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)
print(importances.head(10))

V14    0.179866
V10    0.115520
V12    0.096695
V4     0.096145
V17    0.095122
V3     0.068673
V11    0.056152
V16    0.040224
V2     0.036105
V9     0.026573
dtype: float64


In [ ]:
import joblib

joblib.dump(model, 'fraud_model.pkl')
print("Model saved!")

Model saved!


In [ ]:
import datetime

# Global log to store every decision (this is your audit trail)
audit_log = []

def evaluate_transaction(transaction_row, threshold=0.3):
    """
    Takes ONE transaction (a pandas Series or dict-like row),
    returns a decision with explanation, and logs it.
    """
    try:
        # Convert to the right shape for the model (1 row, all columns)
        features = pd.DataFrame([transaction_row])[X_train.columns]

        # Get fraud probability
        prob = model.predict_proba(features)[0][1]
        decision = "FLAGGED" if prob >= threshold else "PASS"

        # Explain using top features (simple version: show their values)
        top_features = importances.head(3).index.tolist()
        reason = {feat: float(transaction_row[feat]) for feat in top_features}

        result = {
            "timestamp": str(datetime.datetime.now()),
            "decision": decision,
            "fraud_probability": round(float(prob), 4),
            "threshold_used": threshold,
            "top_signals": reason,
            "status": "OK"
        }

    except Exception as e:
        # GRACEFUL FAILURE: if something's wrong with the input,
        # don't crash — flag for manual review instead
        result = {
            "timestamp": str(datetime.datetime.now()),
            "decision": "MANUAL_REVIEW",
            "fraud_probability": None,
            "threshold_used": threshold,
            "top_signals": None,
            "status": f"ERROR: {str(e)}"
        }

    audit_log.append(result)
    return result

In [ ]:
# Grab one real test transaction and run it through the agent
sample = X_test.iloc[0]
result = evaluate_transaction(sample)
print(result)

{'timestamp': '2026-08-22 15:38:57.842878', 'decision': 'PASS', 'fraud_probability': 0.0, 'threshold_used': 0.3, 'top_signals': {'V14': 0.266371326, 'V10': 0.650757004, 'V12': -0.229961446}, 'status': 'OK'}


In [ ]:
# Simulate a corrupted transaction (missing a required field)
broken_transaction = X_test.iloc[1].drop('V14')  # remove a column on purpose

result = evaluate_transaction(broken_transaction)
print(result)

{'timestamp': '2026-08-22 15:39:42.727693', 'decision': 'MANUAL_REVIEW', 'fraud_probability': None, 'threshold_used': 0.3, 'top_signals': None, 'status': 'ERROR: "[\'V14\'] not in index"'}


In [ ]:
# Take a batch of 60 test transactions
batch = X_test.iloc[:60].copy()
true_labels = y_test.iloc[:60].copy()

batch_results = []
for idx in range(len(batch)):
    row = batch.iloc[idx]
    result = evaluate_transaction(row)
    result['true_label'] = int(true_labels.iloc[idx])  # 0 or 1, the real answer
    batch_results.append(result)

results_df = pd.DataFrame(batch_results)
print(results_df[['decision', 'fraud_probability', 'status', 'true_label']])

   decision  fraud_probability status  true_label
0      PASS                0.0     OK           0
1      PASS                0.0     OK           0
2      PASS                0.0     OK           0
3      PASS                0.0     OK           0
4      PASS                0.0     OK           0
5      PASS                0.0     OK           0
6      PASS                0.0     OK           0
7      PASS                0.0     OK           0
8      PASS                0.0     OK           0
9      PASS                0.0     OK           0
10     PASS                0.0     OK           0
11     PASS                0.0     OK           0
12     PASS                0.0     OK           0
13     PASS                0.0     OK           0
14     PASS                0.0     OK           0
15     PASS                0.0     OK           0
16     PASS                0.0     OK           0
17     PASS                0.0     OK           0
18     PASS                0.0     OK           0
